# 板块七：nn 模块 —— 构建神经网络的核心工具箱
torch.nn 是 PyTorch 提供的神经网络模块库，它封装了：  
  
网络层（Linear, Conv2d, LSTM...）  
激活函数（ReLU, Sigmoid...）  
损失函数（CrossEntropyLoss, MSELoss...）  
容器（Sequential, ModuleList...）  
使用 nn，你可以用几行代码搭建复杂模型，而无需手动管理权重和梯度。  
  
我们将按大纲顺序，逐一讲解四大核心组件。  

## 一、网络层（Layers）—— 模型的“积木”

### 1. 全连接层：nn.Linear

In [12]:
import torch
import torch.nn as nn

# 定义：输入10维，输出5维
linear = nn.Linear(in_features=10, out_features=5)

# 查看参数（自动初始化，requires_grad=True）
print("Weight shape:", linear.weight.shape)  # (5, 10)
print("Bias shape:", linear.bias.shape)      # (5,)

# 使用
x = torch.randn(3, 10)  # batch_size=3
output = linear(x)      # 自动计算: x @ weight.T + bias
print("Output shape:", output.shape)  # (3, 5)

Weight shape: torch.Size([5, 10])
Bias shape: torch.Size([5])
Output shape: torch.Size([3, 5])


### 2. 卷积层：nn.Conv2d

In [13]:
# 输入通道=3（RGB），输出通道=16，卷积核=3×3，padding=1（保持尺寸）
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

x = torch.randn(1, 3, 32, 32)  # (batch, C, H, W)
feat = conv(x)
print("Feature map shape:", feat.shape)  # (1, 16, 32, 32)

Feature map shape: torch.Size([1, 16, 32, 32])


### 3. 其他常用层

In [14]:
# Dropout（训练时随机丢弃神经元，防过拟合）
dropout = nn.Dropout(p=0.5)

# BatchNorm（加速训练，稳定分布）
bn = nn.BatchNorm1d(num_features=10)

# 嵌入层（用于词向量）
embedding = nn.Embedding(num_embeddings=1000, embedding_dim=128)
input_ids = torch.tensor([10, 20, 30])
emb = embedding(input_ids)  # (3, 128)

## 二、激活函数（Activation Functions）
非线性激活让网络能拟合复杂函数。PyTorch 提供函数形式和模块形式。

### 模块形式（推荐用于 nn.Sequential）


In [15]:
relu = nn.ReLU()
sigmoid = nn.Sigmoid()
tanh = nn.Tanh()
softmax = nn.Softmax(dim=1)  # dim=1 表示对类别维度归一化

x = torch.tensor([-1.0, 0.0, 1.0, 2.0])
print("ReLU:", relu(x))        # [0., 0., 1., 2.]
print("Sigmoid:", sigmoid(x))  # [0.2689, 0.5, 0.7311, 0.8808]

ReLU: tensor([0., 0., 1., 2.])
Sigmoid: tensor([0.2689, 0.5000, 0.7311, 0.8808])


### 函数形式（更灵活）

In [16]:
import torch.nn.functional as F

y = F.relu(x)  # 等价于 nn.ReLU()(x)

## 三、损失函数（Loss Functions）
损失函数衡量预测与真实值的差距，必须是标量，用于 .backward()。

### 分类任务：交叉熵损失 nn.CrossEntropyLoss

In [17]:
# 输入：logits（未归一化的分数），形状 (N, C)
# 目标：类别索引，形状 (N,)
criterion = nn.CrossEntropyLoss()

logits = torch.tensor([[2.0, 1.0, 0.1],   # 第0类得分最高
                       [0.5, 3.0, 1.2]])  # 第1类得分最高
targets = torch.tensor([0, 1])            # 真实标签

loss = criterion(logits, targets)
print("CrossEntropy loss:", loss.item())  # ~0.23

# 内部自动做了 softmax + 负对数似然

CrossEntropy loss: 0.3190392255783081


### 回归任务：均方误差 nn.MSELoss

In [18]:
pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.5, 2.5, 3.5])

mse_loss = nn.MSELoss()
loss = mse_loss(pred, target)
print("MSE loss:", loss.item())  # 0.25

MSE loss: 0.25


| 任务           | 损失函数                                      |
|----------------|-----------------------------------------------|
| 二分类         | `nn.BCEWithLogitsLoss()`（推荐，含 sigmoid）   |
| 多标签分类     | `nn.BCELoss()`（需先 sigmoid）                |
| 对比学习       | `nn.CosineEmbeddingLoss()`                    |

## 四、优化器（Optimizers）—— 参数更新器
优化器根据梯度更新模型参数。

In [23]:
x = torch.randn(100, 10)      
model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
# 或更常用的 Adam
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 训练循环
for epoch in range(100):
    optimizer.zero_grad()       # 1. 清零梯度
    output = model(x)           # 2. 前向
    loss = criterion(output, y) # 3. 计算损失
    loss.backward()             # 4. 反向传播
    optimizer.step()            # 5. 更新参数

## 五、综合实战：用 nn.Module 自定义模型

In [24]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.layers(x)

# 使用
model = SimpleMLP(10, 32, 1)
x = torch.randn(5, 10)
y_pred = model(x)
print("Prediction shape:", y_pred.shape)  # (5, 1)

Prediction shape: torch.Size([5, 1])


## 六、动手练习：搭建一个图像分类小网络

In [25]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32x32 → 16x16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16 → 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 测试
model = CNN(num_classes=10)
dummy_input = torch.randn(1, 3, 32, 32)
output = model(dummy_input)
print("Output logits shape:", output.shape)  # (1, 10)

Output logits shape: torch.Size([1, 10])


| 组件       | 关键类/函数                     | 用途               |
|------------|----------------------------------|--------------------|
| 层         | `nn.Linear`, `nn.Conv2d`        | 构建网络结构       |
| 激活       | `nn.ReLU()`, `F.relu`           | 引入非线性         |
| 损失       | `nn.CrossEntropyLoss`           | 计算训练目标       |
| 优化器     | `torch.optim.Adam`              | 更新参数           |
| 自定义模型 | `class MyModel(nn.Module)`      | 灵活搭建           |